In [1]:
import ROOT
import pandas as pd
import numpy as np
import os
import sys
import ipynbname
from pathlib import Path

project_root = str(ipynbname.path().parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

from processing import SiphraAcquisition, MetadataLoader, MatchedSiphraAcquisition, match_events
from analysis import fit_peak_expbg

# Constants
BITS12 = 2**12
BITS9 = 2**9 # 512 typical number of bins used

# Energy emission peaks in MeV
K40_MEV = [1.460]
TL208_MEV = [2.614]
CS137_MEV = [0.661]
ANNIHIL_MEV = 0.511
NA22_MEV = [ANNIHIL_MEV, 1.2745, 1.2745+ANNIHIL_MEV]
CO60_MEV = [(1.1732+1.3325)/2, 1.1732+1.3325] # First value is the average of the two photopeaks. They have the same branching ratio


colors = [ROOT.kRed, ROOT.kBlue, ROOT.kGreen, ROOT.kOrange, ROOT.kViolet, ROOT.kYellow, ROOT.kSpring, ROOT.kCyan,]

In [2]:
files_A = sorted((project_root/'data/BigBeautifulDataset_260609').glob('A_SUBT_[3, 4, 5, 6, 7]*.csv'))
files_B = sorted((project_root/'data/BigBeautifulDataset_260609').glob('B_SUBT_[3, 4, 5, 6, 7]*.csv'))

acqs_A = [SiphraAcquisition(file) for file in files_A]
acqs_B = [SiphraAcquisition(file) for file in files_B]
print(files_A)
print(files_B)

[PosixPath('/media/sf_MScThesis/Python_Scripts/SIPHRA_Testing/data/BigBeautifulDataset_260609/A_SUBT_3_Cs137OnCrystalAtCenter_DoubleSiphra.csv'), PosixPath('/media/sf_MScThesis/Python_Scripts/SIPHRA_Testing/data/BigBeautifulDataset_260609/A_SUBT_4_Co60OnCrystalAtCenter_DoubleSiphra.csv'), PosixPath('/media/sf_MScThesis/Python_Scripts/SIPHRA_Testing/data/BigBeautifulDataset_260609/A_SUBT_5_Na22OnCrystalAtCenter_DoubleSiphra.csv'), PosixPath('/media/sf_MScThesis/Python_Scripts/SIPHRA_Testing/data/BigBeautifulDataset_260609/A_SUBT_6_Cs137OnCrystalAtCenterCollimated_DoubleSiphra.csv'), PosixPath('/media/sf_MScThesis/Python_Scripts/SIPHRA_Testing/data/BigBeautifulDataset_260609/A_SUBT_7_BackgroundAtCloseout_DoubleSiphra.csv')]
[PosixPath('/media/sf_MScThesis/Python_Scripts/SIPHRA_Testing/data/BigBeautifulDataset_260609/B_SUBT_3_Cs137OnCrystalAtCenter_DoubleSiphra.csv'), PosixPath('/media/sf_MScThesis/Python_Scripts/SIPHRA_Testing/data/BigBeautifulDataset_260609/B_SUBT_4_Co60OnCrystalAtCente

In [3]:

file_names = ["MATCHED_SUBT_3_Cs137OnCrystalAtCenter_DoubleSiphra", "MATCHED_SUBT_4_Co60OnCrystalAtCenter_DoubleSiphra", 
              "MATCHED_SUBT_5_Na22OnCrystalAtCenter_DoubleSiphra", "MATCHED_SUBT_6_Cs137OnCrystalAtCenterCollimated_DoubleSiphra", 
              "MATCHED_SUBT_7_BackgroundAtCloseout_DoubleSiphra"]

#ONLY RUN THIS ONCE!
"""for i, (file_B, file_A) in enumerate(zip(files_B, files_A)):
    match_events(file_B, file_A, output_path=f"../data/BigBeautifulDataset_260609/{file_names[i]}.csv", tolerance = 100)"""

files_matched = sorted((project_root/'data/BigBeautifulDataset_260609').glob('MATCHED_SUBT_[3, 4, 5, 6, 7]*.csv'))
acqs_matched = [MatchedSiphraAcquisition(file) for file in files_matched]
print(acqs_matched)

[MatchedSiphraAcquisition(File='MATCHED_SUBT_3_Cs137OnCrystalAtCenter_DoubleSiphra'), MatchedSiphraAcquisition(File='MATCHED_SUBT_4_Co60OnCrystalAtCenter_DoubleSiphra'), MatchedSiphraAcquisition(File='MATCHED_SUBT_5_Na22OnCrystalAtCenter_DoubleSiphra'), MatchedSiphraAcquisition(File='MATCHED_SUBT_6_Cs137OnCrystalAtCenterCollimated_DoubleSiphra'), MatchedSiphraAcquisition(File='MATCHED_SUBT_7_BackgroundAtCloseout_DoubleSiphra')]


In [12]:
# histograms
hists = {}

acqs = [acqs_A, acqs_B, acqs_matched]
sources = ['Cs-137', 'Co-60', 'Na-22', 'Cs-137_collimated', ]
#sources = []

siphras = ['A', 'B', 'Summed']

for i, (siphra, acq), in enumerate(zip(siphras, acqs)):
    bg = acq[4]
    #sources = []

    for j in range(4):
        sgnl = acq[j]
        src = sources[j]
        filepath = sgnl.filepath.stem
        #src = (MetadataLoader.load(sgnl.metadataFile)).source
        print(src)
        #sources.append(src)
        # Signal and Background
        hist_sgnl = ROOT.TH1F(f"{src} Signal, SIPHRA {siphra}", "", BITS12, 0, BITS12)
        hist_bg = ROOT.TH1F(f"{src} Background, SIPHRA {siphra}", "", BITS12, 0 , BITS12)
        if siphra == 'Summed':
            hist_sgnl.Fill(sgnl['Combined_Summed']/32)
            hist_bg.Fill(bg['Combined_Summed']/32)
        else:
            hist_sgnl.Fill(sgnl['s']/16)
            hist_bg.Fill(bg['s']/16)
        hist_bg.Scale(sgnl.exposure/bg.exposure)
        # Clean spectrum
        hist_clean = hist_sgnl.Clone(f"{src} BG subtracted, SIPHRA {siphra}")
        hist_clean.Add(hist_bg, -1)
        for hist in [hist_sgnl, hist_bg, hist_clean]:
            # hist.SetTitle(r"^{22}Na spectra - CI gain = 1/0.75 pC")
            hist.GetXaxis().SetTitle("ADC channel number")
            hist.GetYaxis().SetTitle(r"Normalized counts")
        hists[f"{src}_SIPHRA_{siphra}"] = hist_sgnl
        hists[f"{src}_BG_SIPHRA_{siphra}"] = hist_bg
        hists[f"{src}_clean_SIPHRA_{siphra}"] = hist_clean
        del hist_sgnl, hist_bg

Cs-137
Co-60
Na-22
Cs-137_collimated
Cs-137
Co-60
Na-22
Cs-137_collimated
Cs-137
Co-60
Na-22
Cs-137_collimated


In [13]:
if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

cv = ROOT.TCanvas('cv', 'cv', 3200, 2400)
cv.Divide(4, 3)

ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lgds = [ROOT.TLegend(0.48, 0.61, 0.75, 0.83) for _ in range(13)]
print(sources)

for i, siphra in enumerate(siphras):
    for idx, src in enumerate(sources):   
        cv.cd(4*i + idx+1)
        lg = lgds[4*i + idx]
        cv.cd(4*i + idx+1).SetLogy()

        print(src)
        print(f"{i}, {idx}")
        
        sgnl = hists[f"{src}_SIPHRA_{siphra}"] 
        bg = hists[f"{src}_BG_SIPHRA_{siphra}"] 
        clean = hists[f"{src}_clean_SIPHRA_{siphra}"]
        lg.AddEntry(sgnl, "Signal")
        lg.AddEntry(bg, "Background")
        lg.AddEntry(clean, "Subtracted")
        sgnl.SetLineColor(colors[0])
        bg.SetLineColor(colors[1])
        clean.SetLineColor(colors[2])
        if siphra == "Summed":
            sgnl.SetTitle(f"{src}, Summed detector A and B")
        else:
            sgnl.SetTitle(f"{src}, Detector setup {siphra}")
        sgnl.Draw("hist")
        bg.Draw("sames hist")
        clean.Draw("sames hist")
        ROOT.gPad.Update()
        lg.Draw()
        
        cv.Draw()

['Cs-137', 'Co-60', 'Na-22', 'Cs-137_collimated']
Cs-137
0, 0
Co-60
0, 1
Na-22
0, 2
Cs-137_collimated
0, 3
Cs-137
1, 0
Co-60
1, 1
Na-22
1, 2
Cs-137_collimated
1, 3
Cs-137
2, 0
Co-60
2, 1
Na-22
2, 2
Cs-137_collimated
2, 3


In [6]:
from analysis import*

src = "Co-60"
siphra = "A"
hist = hists[f"{src}_clean_SIPHRA_{siphra}"]
ranges = [(160,320), (520,630),(720,870)]

range_Na_B = [(160, 300), (470, 590), (650, 800)]
range_Na_sum =[(180, 270), (500, 615), (690, 840)]
range_Cs_sum = [(250, 370)]
range_Cs_B = [(210, )]
range_Co_B = [(940, 1100)]
range_Co_sum = [(1000, 1200)]
range_Co_A = [(1030, 1220)]


energy_resolution(hist, range_Co_A)


Resolution: 12.9 %
Resolution Error: 0.128 %


[0.12866956128949475]

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      191.224
NDf                       =          187
Edm                       =  2.97963e-07
NCalls                    =           79
Constant                  =      255.868   +/-   2.0329      
Mean                      =      1128.61   +/-   0.483617    
Sigma                     =      61.7948   +/-   0.61315      	 (limited)


In [14]:
from analysis import *

#Calibration based on the 3 peaks of the Na-22 spectrum
#siphra = "Summed"
src = "Na-22"
c_fits = []
for siphra in siphras:
    hist = hists[f"{src}_clean_SIPHRA_{siphra}"]
    energy_ranges = [(180, 270), (500, 615), (690, 840)]
    energies = NA22_MEV
    
    c_fit = calibration_fit(hist, energy_ranges, energies)
    c_fits.append(c_fit)
    print(c_fit)
    

[0.0022985875934603143, -0.04879600680677164]
[0.002477785017622532, -0.03648433358724749]
[0.0023689194210451147, -0.030485452583000176]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      114.124
NDf                       =           87
Edm                       =   3.0676e-06
NCalls                    =          104
Constant                  =      1879.58   +/-   7.65802     
Mean                      =       239.16   +/-   0.413453    
Sigma                     =       47.838   +/-   0.599513     	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      183.695
NDf                       =          112
Edm                       =  9.31736e-09
NCalls                    =           78
Constant                  =      676.269   +/-   4.20464     
Mean                      =      587.599   +/-   0.820404    
Sigma                     =      49.9423   +/-   0.787162     	 (limit

In [8]:
#Calibrating Cs137 histograms based on Na22 calibration fit
c_fit = c_fits[0]

hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[0][4], BITS12, title = r"^{137}Cs 2-sided setup detector A")
hist_cal_bg_Cs137.Scale(1/acqs[0][4].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[0][3], BITS12, title = r"^{137}Cs 2-sided setup detector A")
hist_cal_src_Cs137.Scale(1/acqs[0][3].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
#hist_cal_bg_Cs137.SetTitle(r" ^{137}Cs, 2-sided setup detector A")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{137}Cs signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{137}Cs background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()





Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).


In [16]:
#Calibrating Cs137 histograms based on Na22 calibration fit

source = 1
siphra = 0
background = 4
c_fit = c_fits[siphra]

print(acqs[siphra][source].active_chs)

summed = False
if siphra ==2:
    summed = True

hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[siphra][background], BITS12, title = r"^{137}Cs 2-sided summed setup", summed=summed)
hist_cal_bg_Cs137.Scale(1/acqs[siphra][background].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[siphra][source], BITS12, title = r"^{137}Cs 2-sided summed setup", summed=summed)
hist_cal_src_Cs137.Scale(1/acqs[siphra][source].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{137}Cs signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{137}Cs background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()


[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]


Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).


In [10]:
#Calibrating Cs137 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[2][0], BITS12, title = r"^{137}Cs 2-sided setup summed", summed = True)
hist_cal_bg_Cs137.Scale(1/acqs[2][0].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[2][1], BITS12, title = r"^{137}Cs 2-sided setup summed", summed = True)
hist_cal_src_Cs137.Scale(1/acqs[2][1].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{137}Cs signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{137}Cs background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()

#Calculate energy resolution
res_Cs137 = energy_resolution(hist_cal_clean_Cs137, [(0.5, 0.75)])
print(res_Cs137)


ReferenceError: attempt to access a null-pointer

Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <ROOT::Fit::FillData>: fit range is outside histogram range, no fit data for xaxis
Warning in <Fit>: Fit data is empty 
Error in <TFitResultPtr>: TFitResult is empty - use the fit option S


In [17]:

#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[0][4], BITS12, title = r"^{22}Na 2-sided setup detector A")
hist_cal_bg_Cs137.Scale(1/acqs[0][4].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[0][2], BITS12, title = r"^{22}Na 2-sided setup detector A")
hist_cal_src_Cs137.Scale(1/acqs[0][2].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{22}Na signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{22}Na background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()


Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).


In [18]:

#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[1][4], BITS12, title = r"^{22}Na 2-sided setup detector B")
hist_cal_bg_Cs137.Scale(1/acqs[1][4].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[1][2], BITS12, title = r"^{22}Na 2-sided setup detector B")
hist_cal_src_Cs137.Scale(1/acqs[1][2].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{22}Na signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{22}Na background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()


Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).


In [19]:

#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[2][4], BITS12, title = r"^{22}Na 2-sided setup summed A and B", summed = True)
hist_cal_bg_Cs137.Scale(1/acqs[2][4].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[2][2], BITS12, title = r"^{22}Na 2-sided setup summed A and B", summed = True)
hist_cal_src_Cs137.Scale(1/acqs[2][2].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{22}Na signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{22}Na background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()


Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).


In [20]:

#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[0][4], BITS12, title = r"^{60}Co 2-sided setup detector A")
hist_cal_bg_Cs137.Scale(1/acqs[0][4].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[0][1], BITS12, title = r"^{60}Co 2-sided setup detector A")
hist_cal_src_Cs137.Scale(1/acqs[0][1].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{60}Co signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{60}Co background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()

Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).


In [22]:
#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[1][4], BITS12, title = r"^{60}Co 2-sided setup detector B")
hist_cal_bg_Cs137.Scale(1/acqs[1][4].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[1][1], BITS12, title = r"^{60}Co 2-sided setup detector B")
hist_cal_src_Cs137.Scale(1/acqs[1][1].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{60}Co signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{60}Co background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()


Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).


In [24]:
#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[2][4], BITS12, title = r"^{60}Co 2-sided setup summed A and B", summed = True)
hist_cal_bg_Cs137.Scale(1/acqs[2][4].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[2][1], BITS12, title = r"^{60}Co 2-sided setup summed A and B", summed = True)
hist_cal_src_Cs137.Scale(1/acqs[2][1].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)

ROOT.gStyle.SetOptStat(0)

# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1200, 800)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
hist_cal_bg_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.SetLineColor(colors[0])
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"^{60}Co signal", "l")
lg1.AddEntry(hist_cal_clean_Cs137, r"^{60}Co background subtracted", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
hist_cal_clean_Cs137.Draw("sames hist")
lg1.Draw()
cv1.SetLogy()
cv1.Draw()


Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
